In [ ]:
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

import sklearn
print("Scikit-learn version:", sklearn.__version__)

TensorFlow version: 2.21.0
Scikit-learn version: 1.7.0


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv("/content/archive (11).zip")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

Column Names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Statistical Summary:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000



Missing Values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (768, 8)
y shape: (768,)

Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


In [ ]:
zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[zero_columns] = X[zero_columns].replace(0, np.nan)

print("Missing values after replacing invalid zeros:")
print(X.isnull().sum())

Missing values after replacing invalid zeros:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64


In [ ]:
X = X.fillna(X.median())

print("Missing values after median imputation:")
print(X.isnull().sum())

Missing values after median imputation:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

Training data: (614, 8)
Testing data : (154, 8)


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Feature scaling completed successfully.")

Feature scaling completed successfully.


In [ ]:
class DiabetesNN(ClassifierMixin, BaseEstimator):

    def __init__(
        self,
        learning_rate=0.001,
        init_mode="glorot_uniform",
        activation="relu",
        dropout_rate=0.0,
        neurons=16,
        batch_size=32,
        epochs=50,
        random_state=42
    ):
        self.learning_rate = learning_rate
        self.init_mode = init_mode
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.neurons = neurons
        self.batch_size = batch_size
        self.epochs = epochs
        self.random_state = random_state

    def _build_model(self, input_dim):

        model = Sequential()

        # Hidden Layer
        model.add(
            Dense(
                units=self.neurons,
                activation=self.activation,
                kernel_initializer=self.init_mode,
                input_shape=(input_dim,)
            )
        )

        # Dropout Layer
        if self.dropout_rate > 0:
            model.add(
                Dropout(self.dropout_rate)
            )

        # Output Layer
        model.add(
            Dense(
                units=1,
                activation="sigmoid"
            )
        )

        optimizer = Adam(
            learning_rate=self.learning_rate
        )

        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model

    def fit(self, X, y):

        # Set random seeds
        np.random.seed(self.random_state)
        tf.random.set_seed(self.random_state)

        # Store classes
        self.classes_ = np.unique(y)

        # Build model
        self.model_ = self._build_model(
            X.shape[1]
        )

        # Train model
        self.model_.fit(
            X,
            y,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=0
        )

        return self

    def predict_proba(self, X):

        probabilities = self.model_.predict(
            X,
            verbose=0
        ).ravel()

        return np.column_stack(
            [
                1 - probabilities,
                probabilities
            ]
        )

    def predict(self, X):

        probabilities = self.predict_proba(X)[:, 1]

        return (
            probabilities >= 0.5
        ).astype(int)

    def score(self, X, y):

        predictions = self.predict(X)

        return accuracy_score(
            y,
            predictions
        )

In [ ]:
test_model = DiabetesNN(
    neurons=16,
    epochs=5,
    batch_size=32
)

test_model.fit(
    X_train,
    y_train
)

test_predictions = test_model.predict(
    X_test
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("Test Model Accuracy:", test_accuracy)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  


Test Model Accuracy: 0.6103896103896104


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_batch_epochs = {
    "batch_size": [16, 32],
    "epochs": [50, 100]
}

grid_batch_epochs = GridSearchCV(
    estimator=model,
    param_grid=param_grid_batch_epochs,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_batch_epochs.fit(
    X_train,
    y_train
)

print("Best Batch Size and Epochs:")
print(grid_batch_epochs.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_batch_epochs.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Batch Size and Epochs:
{'batch_size': 32, 'epochs': 100}

Best Cross-Validation Accuracy:
0.7899250757213455


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_learning_rate = {
    "learning_rate": [
        0.001,
        0.01,
        0.1
    ]
}

grid_learning_rate = GridSearchCV(
    estimator=model,
    param_grid=param_grid_learning_rate,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_learning_rate.fit(
    X_train,
    y_train
)

print("Best Learning Rate:")
print(grid_learning_rate.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_learning_rate.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Learning Rate:
{'learning_rate': 0.001}

Best Cross-Validation Accuracy:
0.7866650725330784


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_initialization = {
    "init_mode": [
        "glorot_uniform",
        "he_uniform"
    ]
}

grid_initialization = GridSearchCV(
    estimator=model,
    param_grid=param_grid_initialization,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_initialization.fit(
    X_train,
    y_train
)

print("Best Weight Initialization:")
print(grid_initialization.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_initialization.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Weight Initialization:
{'init_mode': 'glorot_uniform'}

Best Cross-Validation Accuracy:
0.7899250757213455


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_activation = {
    "activation": [
        "relu",
        "tanh",
        "sigmoid"
    ]
}

grid_activation = GridSearchCV(
    estimator=model,
    param_grid=param_grid_activation,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_activation.fit(
    X_train,
    y_train
)

print("Best Activation Function:")
print(grid_activation.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_activation.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Activation Function:
{'activation': 'sigmoid'}

Best Cross-Validation Accuracy:
0.7784871672246134


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_dropout = {
    "dropout_rate": [
        0.0,
        0.2,
        0.4,
        0.5
    ]
}

grid_dropout = GridSearchCV(
    estimator=model,
    param_grid=param_grid_dropout,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_dropout.fit(
    X_train,
    y_train
)

print("Best Dropout Rate:")
print(grid_dropout.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_dropout.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Dropout Rate:
{'dropout_rate': 0.0}

Best Cross-Validation Accuracy:
0.780161007492428


In [ ]:
model = DiabetesNN(
    random_state=42
)

param_grid_neurons = {
    "neurons": [
        8,
        16,
        32,
        64
    ]
}

grid_neurons = GridSearchCV(
    estimator=model,
    param_grid=param_grid_neurons,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

grid_neurons.fit(
    X_train,
    y_train
)

print("Best Number of Hidden Neurons:")
print(grid_neurons.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_neurons.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)`

Best Number of Hidden Neurons:
{'neurons': 32}

Best Cross-Validation Accuracy:
0.7932010202454967


In [ ]:
best_batch_size = grid_batch_epochs.best_params_[
    "batch_size"
]

best_epochs = grid_batch_epochs.best_params_[
    "epochs"
]

best_learning_rate = grid_learning_rate.best_params_[
    "learning_rate"
]

best_init = grid_initialization.best_params_[
    "init_mode"
]

best_activation = grid_activation.best_params_[
    "activation"
]

best_dropout = grid_dropout.best_params_[
    "dropout_rate"
]

best_neurons = grid_neurons.best_params_[
    "neurons"
]

print("=" * 60)
print("             BEST HYPERPARAMETERS")
print("=" * 60)

print("Batch Size           :", best_batch_size)
print("Epochs               :", best_epochs)
print("Learning Rate        :", best_learning_rate)
print("Weight Initialization:", best_init)
print("Activation Function  :", best_activation)
print("Dropout Rate         :", best_dropout)
print("Hidden Neurons       :", best_neurons)

print("=" * 60)

             BEST HYPERPARAMETERS
Batch Size           : 32
Epochs               : 100
Learning Rate        : 0.001
Weight Initialization: glorot_uniform
Activation Function  : sigmoid
Dropout Rate         : 0.0
Hidden Neurons       : 32


In [ ]:
final_model = Sequential()

# Hidden Layer
final_model.add(
    Dense(
        units=best_neurons,
        activation=best_activation,
        kernel_initializer=best_init,
        input_shape=(X_train.shape[1],)
    )
)

# Dropout
if best_dropout > 0:
    final_model.add(
        Dropout(best_dropout)
    )

# Output Layer
final_model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

# Optimizer
final_optimizer = Adam(
    learning_rate=best_learning_rate
)

# Compile
final_model.compile(
    optimizer=final_optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

final_model.summary()

Model: "sequential_67"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_134 (Dense)               │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_135 (Dense)               │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = final_model.fit(
    X_train,
    y_train,
    epochs=best_epochs,
    batch_size=best_batch_size,
    validation_split=0.20,
    verbose=1
)

In [ ]:
test_loss, test_accuracy = final_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("=" * 50)
print("FINAL MODEL RESULTS")
print("=" * 50)

print("Test Loss    :", test_loss)
print("Test Accuracy:", test_accuracy)

